# 3. Transformer Model: DistilBERT (via Hugging Face `Trainer`)

We fine-tune DistilBERT twice: once for **category**, once for **urgency**
 as two independent single-task models.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, f1_score
from datasets import Dataset
from transformers import (
    DistilBertTokenizerFast, DistilBertForSequenceClassification,
    TrainingArguments, Trainer,
)
import pickle

DATA_DIR = Path.cwd().parent / 'data'
MODEL_DIR = Path.cwd().parent / 'models'
BASE_MODEL = 'distilbert-base-uncased'
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE)

SAMPLE_SIZE = None if DEVICE == 'cuda' else 4000
MAX_LENGTH = 128 if DEVICE == 'cuda' else 64
EPOCHS_DEFAULT = 3 if DEVICE == 'cuda' else 2
print(f'SAMPLE_SIZE={SAMPLE_SIZE}  MAX_LENGTH={MAX_LENGTH}  EPOCHS_DEFAULT={EPOCHS_DEFAULT}')

df = pd.read_csv(DATA_DIR / 'cfpb_clean.csv')
tokenizer = DistilBertTokenizerFast.from_pretrained(BASE_MODEL)



Device: cpu
SAMPLE_SIZE=4000  MAX_LENGTH=64  EPOCHS_DEFAULT=2


In [2]:
def tokenize_fn(batch):
    return tokenizer(batch['text'], truncation=True, padding='max_length', max_length=MAX_LENGTH)


class WeightedTrainer(Trainer):
    """Trainer with a class-weighted loss, so rare labels aren't drowned
    out by the majority class (which is what produces high accuracy but
    very low macro F1)."""
    def __init__(self, *args, class_weights=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop('labels')
        outputs = model(**inputs)
        logits = outputs.logits
        loss_fct = torch.nn.CrossEntropyLoss(weight=self.class_weights.to(logits.device))
        loss = loss_fct(logits, labels)
        return (loss, outputs) if return_outputs else loss


def finetune_classifier(df, label_col, save_name, epochs=None, batch_size=16):
    if epochs is None:
        epochs = EPOCHS_DEFAULT

    encoder = LabelEncoder()
    df = df.copy()
    df['label'] = encoder.fit_transform(df[label_col])

    # Stratified subsample (per-task, on this task's own label column) so
    # rare classes keep roughly their original share instead of being
    # wiped out by a plain random sample.
    if SAMPLE_SIZE is not None and len(df) > SAMPLE_SIZE:
        df, _ = train_test_split(
            df, train_size=SAMPLE_SIZE, stratify=df['label'], random_state=42
        )
        print(f'Stratified subsample for {label_col}: {len(df)} rows.')

    train_df, temp_df = train_test_split(df, test_size=0.3, stratify=df['label'], random_state=42)
    val_df, test_df = train_test_split(temp_df, test_size=0.5, stratify=temp_df['label'], random_state=42)

    train_ds = Dataset.from_pandas(train_df[['text', 'label']]).map(tokenize_fn, batched=True)
    val_ds = Dataset.from_pandas(val_df[['text', 'label']]).map(tokenize_fn, batched=True)
    test_ds = Dataset.from_pandas(test_df[['text', 'label']]).map(tokenize_fn, batched=True)

    # Inverse-frequency class weights, computed from the training split.
    class_counts = train_df['label'].value_counts().sort_index()
    weights = (1.0 / class_counts).reindex(range(len(encoder.classes_)), fill_value=0.0)
    weights = weights / weights.sum() * len(encoder.classes_)
    class_weights = torch.tensor(weights.values, dtype=torch.float)

    model = DistilBertForSequenceClassification.from_pretrained(
        BASE_MODEL, num_labels=len(encoder.classes_)
    ).to(DEVICE)

    def compute_metrics(eval_pred):
        logits, labels = eval_pred
        preds = np.argmax(logits, axis=1)
        return {
            'accuracy': accuracy_score(labels, preds),
            'macro_f1': f1_score(labels, preds, average='macro'),
        }

    args = TrainingArguments(
        output_dir=str(MODEL_DIR / f'{save_name}_ckpt'),
        num_train_epochs=epochs,
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=batch_size,
        eval_strategy='epoch',
        save_strategy='no',
        logging_steps=50,
        report_to=[],
        use_cpu=(DEVICE == 'cpu'),
        fp16=(DEVICE == 'cuda'),
    )

    trainer = WeightedTrainer(
        model=model, args=args,
        train_dataset=train_ds, eval_dataset=val_ds,
        compute_metrics=compute_metrics,
        class_weights=class_weights,
    )
    trainer.train()

    test_metrics = trainer.evaluate(test_ds)
    print(f'\n{save_name} test metrics:', test_metrics)

    save_dir = MODEL_DIR / save_name
    save_dir.mkdir(exist_ok=True)
    model.save_pretrained(save_dir)
    tokenizer.save_pretrained(save_dir)
    with open(save_dir / 'label_encoder.pkl', 'wb') as f:
        pickle.dump(encoder, f)

    return {
        'accuracy': test_metrics['eval_accuracy'],
        'macro_f1': test_metrics['eval_macro_f1'],
    }


In [3]:
category_metrics = finetune_classifier(df, label_col='product', save_name='distilbert_category')
category_metrics


Stratified subsample for product: 4000 rows.


Map:   0%|          | 0/2800 [00:00<?, ? examples/s]

Map:   0%|          | 0/600 [00:00<?, ? examples/s]

Map:   0%|          | 0/600 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

...

Epoch,Training Loss,Validation Loss,Accuracy,Macro F1
1,1.473930,1.189767,0.718333,0.410317
2,0.968377,1.096030,0.805000,0.523197


Training Loss,Validation Loss,Epoch,Accuracy,Macro F1
0.968377,1.192261,2,0.788333,0.490755



distilbert_category test metrics: {'eval_loss': 1.192260503768921, 'eval_accuracy': 0.7883333333333333, 'eval_macro_f1': 0.4907552996776495}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'accuracy': 0.7883333333333333, 'macro_f1': 0.4907552996776495}

In [4]:
urgency_metrics = finetune_classifier(df, label_col='urgency', save_name='distilbert_urgency')
urgency_metrics


Stratified subsample for urgency: 4000 rows.


Map:   0%|          | 0/2800 [00:00<?, ? examples/s]

Map:   0%|          | 0/600 [00:00<?, ? examples/s]

Map:   0%|          | 0/600 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

...

Epoch,Training Loss,Validation Loss,Accuracy,Macro F1
1,1.024746,0.901633,0.615000,0.518356
2,0.838357,0.900873,0.671667,0.563710


Training Loss,Validation Loss,Epoch,Accuracy,Macro F1
0.838357,1.000156,2,0.638333,0.499458



distilbert_urgency test metrics: {'eval_loss': 1.0001558065414429, 'eval_accuracy': 0.6383333333333333, 'eval_macro_f1': 0.49945798476677244}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'accuracy': 0.6383333333333333, 'macro_f1': 0.49945798476677244}

In [5]:
pd.DataFrame([
    {'model': 'distilbert_category', **category_metrics},
    {'model': 'distilbert_urgency', **urgency_metrics},
]).to_csv(MODEL_DIR / 'distilbert_results.csv', index=False)
print('Saved distilbert results.')


Saved distilbert results.
